# Partie 4 — Apache Iceberg : snapshots, Time Travel et rollback
## Contexte

Les parties précédentes ont permis de construire :

- **Bronze** : copie des sources (PostgreSQL, CSV, JSON) dans Iceberg ;
- **Silver** : normalisation et contrôles de qualité ;
- **Quarantine** : conservation séparée des données rejetées ;
- **Gold** : indicateurs hospitaliers calculés à partir de Silver.

Dans cette dernière partie, nous allons tester ce qui distingue **une table Iceberg** de simples fichiers Parquet : **l'historique, les mises à jour et la consultation des anciennes versions**.

## Objectifs

1. Démarrer une **nouvelle session Spark** indépendante.
2. Consulter les snapshots des tables Gold **en lecture seule**.
3. Créer une petite table d'essai Iceberg, sans toucher aux données réelles du TP.
4. Réaliser un **INSERT**, un **UPDATE** et un **DELETE**.
5. Lire une ancienne version avec **Time Travel**.
6. Revenir volontairement à un snapshot précédent avec **rollback**.
7. Examiner les fichiers Iceberg stockés dans RustFS.

> **Sécurité :** ce notebook **ne modifie jamais les tables `medical.bronze`, `medical.silver`, `medical.gold` ou `medical.quarantine`**. Il travaille dans le namespace `medical.demo_iceberg`. Chaque exécution complète crée une table d'essai différente.


In [1]:
# 1. Démarrer une nouvelle session Spark
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse-Partie4-Iceberg")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.medical", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.medical.type", "rest")
    .config("spark.sql.catalog.medical.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.medical.warehouse", "s3://medical-warehouse/")
    .config("spark.sql.catalog.medical.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.medical.s3.endpoint", "http://rustfs:9000")
    .config("spark.sql.catalog.medical.s3.path-style-access", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Version de Spark :", spark.version)
print("Catalogue REST :", spark.conf.get("spark.sql.catalog.medical.uri"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 12:33:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 12:33:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/19 12:33:27 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/19 12:33:27 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/10/19 12:33:27 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/10/19 12:33:27 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


Version de Spark : 3.5.5
Catalogue REST : http://iceberg-rest:8181


## 2 — Consulter les tables déjà construites

In [2]:
# 2. Voir les namespaces et les tables Gold
spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

if spark.catalog.tableExists("medical.gold.consultations_par_categorie"):
    spark.sql("SHOW TABLES IN medical.gold").show(truncate=False)
else:
    print("Partie Gold non encore disponible : nous poursuivrons avec une table de démonstration.")

+----------+
|namespace |
+----------+
|bronze    |
|gold      |
|quarantine|
|sandbox   |
|silver    |
+----------+

+---------+---------------------------+-----------+
|namespace|tableName                  |isTemporary|
+---------+---------------------------+-----------+
|gold     |activite_mensuelle         |false      |
|gold     |analyses_par_type          |false      |
|gold     |consultations_par_categorie|false      |
|gold     |patients_par_genre         |false      |
+---------+---------------------------+-----------+



## 3 — Comprendre un snapshot Iceberg

Un **snapshot** représente un état cohérent de la table après une opération enregistrée (*commit*).

Iceberg conserve des métadonnées qui permettent de retrouver les fichiers correspondant à une version donnée.

**Attention :** les anciens états ne sont pas conservés éternellement : une opération ultérieure d'expiration des snapshots peut rendre certaines versions indisponibles.

In [3]:
# 3. Lire les snapshots d'une table Gold, si elle existe
nom_gold = "medical.gold.consultations_par_categorie"

if spark.catalog.tableExists(nom_gold):
    spark.sql(f"""
        SELECT snapshot_id, committed_at, operation
        FROM {nom_gold}.snapshots
        ORDER BY committed_at
    """).show(truncate=False)
else:
    print("Table Gold absente : test de snapshots effectué plus bas sur la table de démonstration.")

+-------------------+----------------------+---------+
|snapshot_id        |committed_at          |operation|
+-------------------+----------------------+---------+
|1127280944782587581|2026-10-19 10:51:27.45|append   |
+-------------------+----------------------+---------+



## 4 — Préparer une table de démonstration isolée

Nous allons simuler trois patients fictifs dans une table destinée uniquement aux tests des fonctionnalités Iceberg.

À chaque **Run All**, un suffixe unique crée une nouvelle table. Ainsi, vous pouvez rejouer ce notebook sans modifier vos anciennes tables ni réinsérer accidentellement les mêmes lignes.

Exemple de nom : `medical.demo_iceberg.patients_historique_a1b2c3d4`.

In [4]:
# 4. Créer un namespace dédié et choisir un nouveau nom de table
from uuid import uuid4

spark.sql("CREATE NAMESPACE IF NOT EXISTS medical.demo_iceberg")

suffixe = uuid4().hex[:8]
nom_court = f"patients_historique_{suffixe}"
table_demo = f"medical.demo_iceberg.{nom_court}"
table_procedure = f"demo_iceberg.{nom_court}"

print("Table de démonstration :", table_demo)

# Pour protéger les tables existantes, nous interdisons un nom déjà occupé.
assert not spark.catalog.tableExists(table_demo), "La table de test existe déjà."

Table de démonstration : medical.demo_iceberg.patients_historique_d918fc2b


## 5 — Créer une table Iceberg vide

La table possède trois colonnes simples : `patient_id`, `age` et `ville`.

Nous utilisons le format Iceberg **v2** et le mode **copy-on-write** pour simplifier la démonstration des modifications et suppressions.

La création d'une table vide n'est pas encore une insertion de données.

In [5]:
# 5. Créer la table de test avec Iceberg
spark.sql(f"""
    CREATE TABLE {table_demo} (
        patient_id STRING,
        age INT,
        ville STRING
    )
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '2',
        'write.update.mode' = 'copy-on-write',
        'write.delete.mode' = 'copy-on-write'
    )
""")

print("Table créée :", table_demo)
spark.sql(f"DESCRIBE TABLE {table_demo}").show(truncate=False)

Table créée : medical.demo_iceberg.patients_historique_d918fc2b
+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|patient_id|string   |NULL   |
|age       |int      |NULL   |
|ville     |string   |NULL   |
+----------+---------+-------+



## 6 — Fonctions utiles pour suivre les versions

La table de métadonnées `.snapshots` contient les identifiants de snapshot.

Nous allons comparer **l'ensemble des identifiants avant et après chaque écriture** pour identifier le nouveau snapshot sans dépendre de l'ordre des timestamps.

In [6]:
# 6. Fonctions pour lire les snapshots et détecter le nouveau commit
def identifiants_snapshots(nom_table):
    lignes = spark.sql(f"SELECT snapshot_id FROM {nom_table}.snapshots").collect()
    return {int(ligne["snapshot_id"]) for ligne in lignes}


def nouveau_snapshot(avant, apres):
    nouveaux = apres - avant
    assert len(nouveaux) == 1, (
        f"Un seul nouveau snapshot était attendu, obtenu : {nouveaux}"
    )
    return nouveaux.pop()


def afficher_patients():
    spark.sql(f"SELECT * FROM {table_demo} ORDER BY patient_id").show(truncate=False)

print("Fonctions prêtes.")

Fonctions prêtes.


## 7 — Première insertion : créer le snapshot S1

Nous insérons deux patients fictifs : `P001` et `P002`.

Cette écriture crée la première version contenant des données. Nous enregistrons son identifiant dans `snapshot_1`.

In [7]:
# 7. Insérer deux patients : état initial
avant_1 = identifiants_snapshots(table_demo)

spark.sql(f"""
    INSERT INTO {table_demo}
    VALUES ('P001', 40, 'Tunis'),
           ('P002', 32, 'Sfax')
""")

apres_1 = identifiants_snapshots(table_demo)
snapshot_1 = nouveau_snapshot(avant_1, apres_1)

print("S1 =", snapshot_1)
afficher_patients()
assert spark.table(table_demo).count() == 2

S1 = 8086704273661171740
+----------+---+-----+
|patient_id|age|ville|
+----------+---+-----+
|P001      |40 |Tunis|
|P002      |32 |Sfax |
+----------+---+-----+



## 8 — Deuxième insertion : créer le snapshot S2

Nous simulons l'arrivée d'un **nouveau lot de données** : le patient `P003`.

Le nouvel état contient maintenant trois patients. Iceberg ne modifie pas seulement un fichier CSV : il enregistre un nouveau snapshot représentant la version de la table.

In [8]:
# 8. Ajouter un troisième patient
avant_2 = identifiants_snapshots(table_demo)

spark.sql(f"""
    INSERT INTO {table_demo}
    VALUES ('P003', 28, 'Sousse')
""")

apres_2 = identifiants_snapshots(table_demo)
snapshot_2 = nouveau_snapshot(avant_2, apres_2)

print("S2 =", snapshot_2)
afficher_patients()
assert spark.table(table_demo).count() == 3

S2 = 1773935941960527491
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P001      |40 |Tunis |
|P002      |32 |Sfax  |
|P003      |28 |Sousse|
+----------+---+------+



## 9 — Time Travel : relire un ancien état

**Time Travel** permet de consulter une version précédente **sans modifier l'état actuel**.

Nous comparons :

- **S1** : deux patients ;
- **S2** : trois patients, après le nouvel INSERT.

La syntaxe SQL est `VERSION AS OF <snapshot_id>`.

In [9]:
# 9. Comparer deux snapshots historiques sans rien modifier
print("=== Ancien état S1 ===")
spark.sql(f"""
    SELECT * FROM {table_demo} VERSION AS OF {snapshot_1}
    ORDER BY patient_id
""").show(truncate=False)

print("=== État S2 ===")
spark.sql(f"""
    SELECT * FROM {table_demo} VERSION AS OF {snapshot_2}
    ORDER BY patient_id
""").show(truncate=False)

assert spark.sql(
    f"SELECT * FROM {table_demo} VERSION AS OF {snapshot_1}"
).count() == 2
assert spark.sql(
    f"SELECT * FROM {table_demo} VERSION AS OF {snapshot_2}"
).count() == 3

print("Time Travel vérifié.")

=== Ancien état S1 ===
+----------+---+-----+
|patient_id|age|ville|
+----------+---+-----+
|P001      |40 |Tunis|
|P002      |32 |Sfax |
+----------+---+-----+

=== État S2 ===
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P001      |40 |Tunis |
|P002      |32 |Sfax  |
|P003      |28 |Sousse|
+----------+---+------+

Time Travel vérifié.


## 10 — UPDATE : modifier un enregistrement

Nous simulons une correction administrative : la ville de `P002` passe de `Sfax` à `Sousse`.

**Iceberg permet de modifier une table même lorsque ses données sont stockées en fichiers Parquet.** Le moteur Spark effectue les écritures nécessaires et Iceberg enregistre le nouvel état.

In [10]:
# 10. Mettre à jour une ligne de la table
avant_3 = identifiants_snapshots(table_demo)

spark.sql(f"""
    UPDATE {table_demo}
    SET ville = 'Sousse'
    WHERE patient_id = 'P002'
""")

apres_3 = identifiants_snapshots(table_demo)
snapshot_3 = nouveau_snapshot(avant_3, apres_3)

print("S3 (après UPDATE) =", snapshot_3)
afficher_patients()

ville_p002 = spark.sql(
    f"SELECT ville FROM {table_demo} WHERE patient_id = 'P002'"
).first()["ville"]
assert ville_p002 == "Sousse"

S3 (après UPDATE) = 8625978130595610173
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P001      |40 |Tunis |
|P002      |32 |Sousse|
|P003      |28 |Sousse|
+----------+---+------+



## 11 — DELETE : supprimer une ligne du nouvel état

Nous supprimons le patient fictif `P001` **uniquement dans cette table de démonstration**.

Nous pouvons toujours consulter les anciennes versions tant que leurs snapshots et fichiers nécessaires sont conservés.

In [11]:
# 11. Supprimer P001 dans la version actuelle
avant_4 = identifiants_snapshots(table_demo)

spark.sql(f"""
    DELETE FROM {table_demo}
    WHERE patient_id = 'P001'
""")

apres_4 = identifiants_snapshots(table_demo)
snapshot_4 = nouveau_snapshot(avant_4, apres_4)

print("S4 (après DELETE) =", snapshot_4)
afficher_patients()

assert spark.table(table_demo).count() == 2
assert spark.sql(
    f"SELECT * FROM {table_demo} WHERE patient_id = 'P001'"
).count() == 0

S4 (après DELETE) = 2177621598992357977
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P002      |32 |Sousse|
|P003      |28 |Sousse|
+----------+---+------+



## 12 — Retrouver les quatre versions

Nous avons maintenant produit quatre snapshots après quatre opérations d'écriture :

| Snapshot | Opération | État de la table |
|---|---|---|
| S1 | INSERT initial | P001, P002 |
| S2 | INSERT nouveau lot | P001, P002, P003 |
| S3 | UPDATE de P002 | P001, P002 corrigé, P003 |
| S4 | DELETE de P001 | P002 corrigé, P003 |

Les identifiants sont attribués par Iceberg et changent à chaque exécution.

In [12]:
# 12. Afficher l'historique des snapshots
spark.sql(f"""
    SELECT snapshot_id, parent_id, committed_at, operation
    FROM {table_demo}.snapshots
    ORDER BY committed_at
""").show(truncate=False)

print("Snapshots enregistrés dans le notebook :")
for nom, identifiant in [
    ("S1", snapshot_1),
    ("S2", snapshot_2),
    ("S3", snapshot_3),
    ("S4", snapshot_4),
]:
    print(nom, "=", identifiant)

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|8086704273661171740|NULL               |2026-10-19 12:35:17.793|append   |
|1773935941960527491|8086704273661171740|2026-10-19 12:35:19.595|append   |
|8625978130595610173|1773935941960527491|2026-10-19 12:35:44.616|overwrite|
|2177621598992357977|8625978130595610173|2026-10-19 12:35:53.473|delete   |
+-------------------+-------------------+-----------------------+---------+

Snapshots enregistrés dans le notebook :
S1 = 8086704273661171740
S2 = 1773935941960527491
S3 = 8625978130595610173
S4 = 2177621598992357977


## 13 — Comparer une ancienne version et la version actuelle

Nous allons vérifier que la version historique **S2** contient toujours les trois patients et l'ancienne ville de `P002`.

Pendant cette consultation historique, l'état courant reste **S4**.

In [13]:
# 13. Comparer S2 à la version courante S4
print("=== Version historique S2 ===")
spark.sql(f"""
    SELECT * FROM {table_demo} VERSION AS OF {snapshot_2}
    ORDER BY patient_id
""").show(truncate=False)

print("=== Version actuelle S4 ===")
afficher_patients()

ancienne_ville = spark.sql(f"""
    SELECT ville FROM {table_demo} VERSION AS OF {snapshot_2}
    WHERE patient_id = 'P002'
""").first()["ville"]

print("Ville P002 en S2 :", ancienne_ville)
assert ancienne_ville == "Sfax"

=== Version historique S2 ===
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P001      |40 |Tunis |
|P002      |32 |Sfax  |
|P003      |28 |Sousse|
+----------+---+------+

=== Version actuelle S4 ===
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P002      |32 |Sousse|
|P003      |28 |Sousse|
+----------+---+------+

Ville P002 en S2 : Sfax


## 14 — ROLLBACK : revenir à l'état S2

Contrairement au **Time Travel**, qui effectue seulement une lecture historique, **rollback modifie le snapshot courant de la table**.

Nous revenons volontairement à **S2**, où il y avait trois patients et où `P002` habitait encore `Sfax`.

> **À ne jamais faire sans validation sur les tables Bronze, Silver ou Gold :** un rollback peut rendre invisibles des modifications plus récentes pour les lecteurs de la table. Ici, cette opération concerne uniquement notre table de démonstration.

In [14]:
# 14. Revenir au snapshot S2 sur notre table de démonstration
resultat_rollback = spark.sql(f"""
    CALL medical.system.rollback_to_snapshot(
        table => '{table_procedure}',
        snapshot_id => {snapshot_2}
    )
""")

resultat_rollback.show(truncate=False)

print("=== Données courantes après rollback ===")
afficher_patients()

lignes_apres_retour = spark.sql(
    f"SELECT patient_id, ville FROM {table_demo}"
).collect()
donnees_apres_retour = {l["patient_id"]: l["ville"] for l in lignes_apres_retour}

assert donnees_apres_retour == {
    "P001": "Tunis",
    "P002": "Sfax",
    "P003": "Sousse",
}, "Le rollback n'a pas restauré l'état attendu S2."

print("Rollback vers S2 réussi.")

+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
|2177621598992357977 |1773935941960527491|
+--------------------+-------------------+

=== Données courantes après rollback ===
+----------+---+------+
|patient_id|age|ville |
+----------+---+------+
|P001      |40 |Tunis |
|P002      |32 |Sfax  |
|P003      |28 |Sousse|
+----------+---+------+

Rollback vers S2 réussi.


## 15 — Observer l'historique des changements de référence
La table de métadonnées `.history` montre les événements de changement de snapshot courant, y compris le rollback.


In [15]:
# 15. Afficher l'historique de la table après le rollback
spark.sql(f"""
    SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor
    FROM {table_demo}.history
    ORDER BY made_current_at
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-10-19 12:35:17.793|8086704273661171740|NULL               |true               |
|2026-10-19 12:35:19.595|1773935941960527491|8086704273661171740|true               |
|2026-10-19 12:35:44.616|8625978130595610173|1773935941960527491|false              |
|2026-10-19 12:35:53.473|2177621598992357977|8625978130595610173|false              |
|2026-10-19 12:37:09.42 |1773935941960527491|8086704273661171740|true               |
+-----------------------+-------------------+-------------------+-------------------+



## 16 — Vérifier les fichiers Parquet référencés

La table `.files` expose les fichiers de données référencés **par l'état courant** de notre table.
Ces fichiers se trouvent dans RustFS, bucket **`medical-warehouse`**. Vous pouvez également visiter la console : http://localhost:9001.

In [17]:
# 16. Consulter les fichiers Iceberg du snapshot courant
spark.sql(f"""
    SELECT file_path, record_count, file_size_in_bytes
    FROM {table_demo}.files
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                                         |record_count|file_size_in_bytes|
+----------------------------------------------------------------------------------------------------------------------------------+------------+------------------+
|s3://medical-warehouse/demo_iceberg/patients_historique_d918fc2b/data/00000-9-a20a7e48-284d-43c4-aa5a-14d987edb11c-0-00001.parquet|1           |909               |
|s3://medical-warehouse/demo_iceberg/patients_historique_d918fc2b/data/00000-2-7540c403-7bef-4fb3-83d4-e56e0f36baa3-0-00001.parquet|1           |901               |
|s3://medical-warehouse/demo_iceberg/patients_historique_d918fc2b/data/00001-3-7540c403-7bef-4fb3-83d4-e56e0f36baa3-0-00001.parquet|1           |895               |
+---------

## 17 — Vérification de sécurité : les tables Gold sont intactes

Nous n'avons utilisé `INSERT`, `UPDATE`, `DELETE` et `CALL rollback_to_snapshot` **que** sur une table `medical.demo_iceberg.*`.

Les tables Bronze, Silver, Gold et Quarantine du projet n'ont été ni remplacées ni supprimées par ce notebook.

In [18]:
# 17. Afficher les tables Gold (lecture seule)
if spark.catalog.tableExists("medical.gold.consultations_par_categorie"):
    spark.sql("SHOW TABLES IN medical.gold").show(truncate=False)
    print("Aucune écriture n'a été effectuée dans Gold.")
else:
    print("Gold n'est pas encore disponible : la démonstration Iceberg reste valide.")

print("La table utilisée pour les tests était :", table_demo)

+---------+---------------------------+-----------+
|namespace|tableName                  |isTemporary|
+---------+---------------------------+-----------+
|gold     |activite_mensuelle         |false      |
|gold     |analyses_par_type          |false      |
|gold     |consultations_par_categorie|false      |
|gold     |patients_par_genre         |false      |
+---------+---------------------------+-----------+

Aucune écriture n'a été effectuée dans Gold.
La table utilisée pour les tests était : medical.demo_iceberg.patients_historique_d918fc2b
